In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from Cyclist_env_RDA import cyclist_env
from time import time
from scipy.linalg import dft
import torch
import numpy as np
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
import shutil
from numpy.fft import fft, ifft

In [ ]:
def ca_cfar_2d(x_db, n_train=(1,7), n_guard=(1,5), pfa=1e-6, convert_from_db=False):
    """
    x_db: (B,1,H,W) input map (dB or linear)
    n_train: (th, tw) training half window size
    n_guard: (gh, gw) guard half window size
    """
    if convert_from_db:
        x = 10 ** (x_db / 10.0)  # dB -> linear
    else:
        x = x_db

    B, C, H, W = x.shape
    th, tw = n_train
    gh, gw = n_guard

    # Total kernel size
    kh = 2 * (th + gh) + 1
    kw = 2 * (tw + gw) + 1

    # Make training mask
    kernel = torch.ones((1,1,kh,kw), device=x.device)
    kernel[:, :, th-gh:th+gh+1, tw-gw:tw+gw+1] = 0  # guard + CUT = 0

    N_train = kernel.sum().item()
    alpha = N_train * (pfa ** (-1.0 / N_train) - 1.0)

    # Convolution = training sum
    x_pad = F.pad(x, (kw//2, kw//2, kh//2, kh//2), mode="reflect")
    train_sum = F.conv2d(x_pad.double(), kernel.double())
    noise_est = train_sum / N_train

    threshold = alpha * noise_est
    detections = x > threshold
    return detections, threshold, noise_est

# ???: tar_thres = -25
# ???: tar_thres = -35  (??? -30)
def thresholder(radar, idx_est, tar_thres = -25, cyvethres = 2.638):
    cy_idx, ve_idx = [], []
    radar = 20*torch.log10(radar)
    for i in range(len(idx_est)):
        rangeofinterest = radar[:5, max(0, idx_est[i]-5):min(idx_est[i]+5, 589)].flatten()
        top2 = rangeofinterest.topk(2).values.mean()
        if top2 > tar_thres:
            cy_idx.append(idx_est[i])
            ve_idx.append(idx_est[i]) 

    cy_idx_cleans = cleansing(cy_idx)
    ve_idx_cleans = cleansing(ve_idx)

    return cy_idx_cleans, ve_idx_cleans
    
def cleansing(idx, tol = 5):
    if idx :
        res = [idx[0]]
        for i in range(1,len(idx)):
            if idx[i]<=idx[i-1]+5:
                continue
            res.append(idx[i])
        return res
    else: return []

In [ ]:
# --- 1. CFAR Improved ---
# ???????????????????????
def ca_cfar_improved(x_db, n_train=(1,7), n_guard=(1,5), pfa=1e-6, convert_from_db=False):
    """
    2?? CA-CFAR (Cell Averaging CFAR)
    x_db: (B, 1, H, W) ????? (???dB)
    ???: (B, 1, H, W) ?Bool????? (True?????)
    """
    # ??
    n_guard = (1,10)
    pfa = 1e-4

    if convert_from_db:
        x = 10 ** (x_db / 10.0)
    else:
        x = x_db

    B, C, H, W = x.shape
    th, tw = n_train
    gh, gw = n_guard

    # ?????? (??????0??????1???)
    kh = 2 * (th + gh) + 1
    kw = 2 * (tw + gw) + 1
    kernel = torch.ones((1, 1, kh, kw), device=x.device, dtype=x.dtype)

    #kernel[:, :, th-gh:th+gh+1, tw-gw:tw+gw+1] = 0
    # ???
    kernel[:, :, th : th + 2*gh + 1, tw : tw + 2*gw + 1] = 0

    N_train = kernel.sum().item()
    
    # ???? alpha ???
    alpha = N_train * (pfa ** (-1.0 / N_train) - 1.0)

    # ???????????????
    # mode='reflect' ????????
    #x_pad = F.pad(x, (kw//2, kw//2, kh//2, kh//2), mode="reflect")
    # ???
    # ???????(H)??circular?(??)??????(W)??replicate?(??????)?????
    # PyTorch?F.pad????1????????????????2????????

    # 1. ?????(??)??????(???)
    pad_w = kw // 2
    x_pad_w = F.pad(x, (pad_w, pad_w, 0, 0), mode="replicate") # ??? "constant"

    # 2. ???????(??)??????(??:??????????)
    pad_h = kh // 2
    x_pad = F.pad(x_pad_w, (0, 0, pad_h, pad_h), mode="circular")
    
    train_sum = F.conv2d(x_pad, kernel)
    noise_est = train_sum / N_train

    threshold_map = alpha * noise_est
    detections = x > threshold_map
    
    return detections

# --- 2. Thresholder Improved ---
# ???????(??)??????????????????
def thresholder_improved(radar_db, detections_bool, tar_thres=-25):
    """
    radar_db:        (H, W) ?2?????? (dB???RD???)
    detections_bool: (H, W) ?2??Bool???? (CFAR???)
    tar_thres:       ????????????? (dB)
    
    ???: ????? [[doppler_idx, range_idx, power], ...]
    """
    # CFAR?True???????????????? (Doppler, Range)
    # as_tuple=False ? [[d, r], [d, r]...] ?????????
    coords = torch.nonzero(detections_bool, as_tuple=False).tolist()
    
    valid_targets = []
    H, W = radar_db.shape
    
    for coord in coords:
        d_idx, r_idx = coord[0], coord[1]
        
        # ???? (ROI) ???
        # ??? radar[:5, ...] ?????????
        # ????????? d_idx ???????????
        d_min = max(0, d_idx - 1)
        d_max = min(H, d_idx + 2) # Python???????????????+2
        r_min = max(0, r_idx - 5)
        r_max = min(W, r_idx + 6)
        
        # ROI??????????
        roi = radar_db[d_min:d_max, r_min:r_max]
        peak_power = torch.max(roi).item()
        
        # ??????
        if peak_power > tar_thres:
            valid_targets.append([d_idx, r_idx, peak_power])
            
    return valid_targets

# --- 3. Cleansing Improved ---
# ?????????????????? (NMS: Non-Maximum Suppression ????)
def cleansing_improved(targets, dist_tol=5, dop_tol=2):
    """
    targets: [[dop_idx, rng_idx, power], ...] ????
    dist_tol: ??????????(???)????????
    dop_tol:  ??????????(?????)????????
    
    ???: ??????????????
    """
    if not targets:
        return []
        
    # ??????????? (?????????????)
    # x[2] ? power
    targets_sorted = sorted(targets, key=lambda x: x[2], reverse=True)
    
    final_targets = []
    
    while targets_sorted:
        # ??????????????
        best = targets_sorted.pop(0)
        final_targets.append(best)
        
        # ??????????????????????????
        # ???????????????????????
        candidates = []
        for other in targets_sorted:
            d_diff = abs(best[0] - other[0])
            r_diff = abs(best[1] - other[1])
            
            # ?????(=????)????
            if not (d_diff <= dop_tol and r_diff <= dist_tol):
                candidates.append(other)
        
        targets_sorted = candidates
        
    return final_targets

# --- 4. ????????? (??????????) ---
def convert_to_physical(d_idx, r_idx, N_trans, sym_duration, lam, current_angle, idx_to_xy_func):
    """
    ?????????????(m)???(m/s)???
    """
    # ????
    vel_res = lam / (2.0 * sym_duration * N_trans)
    if d_idx < N_trans / 2:
        velocity = d_idx * vel_res
    else:
        velocity = (d_idx - N_trans) * vel_res
        
    # ??????? (????????)
    x, y, r = idx_to_xy_func(r_idx, current_angle)
    
    return r, velocity, x, y

In [ ]:
def remove_ghosts_across_angles(all_detections, range_tol=1.0, ang_tol=1.0, vel_tol=0.5):
    """
    ???????????????(??????)???????
    all_detections: [[range, vel, angle, x, y, power], ...] ????
    range_tol: ??????????? [m]
    vel_tol:   ??????????? [m/s]
    """
    if not all_detections:
        return []

    # 1. ????????????? (????????power???)
    # x[5] ? power ??
    sorted_dets = sorted(all_detections, key=lambda x: x[5], reverse=True)
    
    final_unique_objects = []
    
    while sorted_dets:
        # ???????????(??????????????)
        best = sorted_dets.pop(0)
        final_unique_objects.append(best)
        
        # ??????????????????????????????
        # (??????????=????????)
        remaining = []
        for other in sorted_dets:
            r_diff = abs(best[0] - other[0])
            a_diff = abs(best[1] - other[1])
            v_diff = abs(best[4] - other[4])
            
            # ????????????????????????????????????(??)
            if r_diff < range_tol and a_diff < ang_tol:
                continue 
            # ????????????????????????????????????(??)
            if r_diff < range_tol and v_diff < vel_tol:
                continue
            remaining.append(other)
        
        sorted_dets = remaining

    return final_unique_objects

In [ ]:
def stevec(N_ant, angle):
    # ????????????????????????
    resp = (np.arange(N_ant)-N_ant*0.5+0.5).reshape([-1,1])
    resp = np.exp(1j*resp*np.pi*np.sin(angle))
    return np.matrix(resp)

In [ ]:
def MUSIC_method(Y, N_ant):
    Y_music = np.mean(Y, axis=0)
    R_yy = Y_music@np.conjugate(np.transpose(Y_music))
    # print(np.linalg.eig(R_yy)[0])
    U = np.linalg.eig(R_yy)[1][:,5:]

    resp = []
    argm = (np.arange(1800)-900)/100
    for val in argm:
        stv = stevec(N_ant, val*np.pi/180)
        p = stv.T@U
        pp = p*np.conjugate(np.transpose(p))
        resp.append(1/(np.abs(pp)**2).A1)

    M = max(resp)
    est_ang = []
    
    a, b = resp[0][0], resp[1][0]
    for i in range(1,len(argm)-1):
        c = resp[i+1][0]
        if a < b and b > c and b > 0.2 * M:
            est_ang.append(argm[i])
        a, b = b, c
    return est_ang

#angle_list = MUSIC_method(Y)
#print("Estimated angles:", angle_list)

In [ ]:
def MUSIC_method_Debug(eig_val, N_ant, num_signals, resp, argm, M):
    """
    MUSIC??????????????????
    """
    print("\n" + "="*30)
    print("?MUSIC? ???????")
    
    # --- 1. ??????? ---
    # ???????(????)
    eig_abs = np.abs(eig_val)
    print(f"1. ??? (????):\n{eig_abs}")
    
    # ?????????????
    print(f"2. ????? (K): {num_signals}")
    print(f"3. ???????? (M): {M:.2e}")
    print(f"4. ????? (20%): {0.2 * M:.2e}")

    # --- 2. ????? (??) ---
    try:
        """
        # ????????2??? (?: ?????, ?: MUSIC?????)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        
        # === ?????: ????? (Scree Plot) ===
        # ???????????????????????????????
        ax1.plot(range(1, len(eig_abs) + 1), eig_abs, 'bo-', markersize=8)
        ax1.set_title('Eigenvalue Distribution (Scree Plot)')
        ax1.set_xlabel('Index')
        ax1.set_ylabel('Eigenvalue (log scale)')
        ax1.set_yscale('log') # ??????????????????????
        ax1.grid(True, which="both", ls="-", alpha=0.5)
        
        # ??????????????????
        ax1.axvline(x=num_signals + 0.5, color='r', linestyle='--', label=f'Separation (K={num_signals})')
        ax1.legend()
        
        # === ?????: MUSIC????? ===
        # ???????????????????????????
        resp_array = np.array(resp).flatten() # ???1?????
        
        ax2.plot(argm, resp_array, label='MUSIC Spectrum')
        ax2.set_title(f'MUSIC Spectrum (N_ant={N_ant}, K={num_signals})')
        ax2.set_xlabel('Angle [deg]')
        ax2.set_ylabel('Spectrum Power')
        ax2.set_yscale('log') # ????????????????????????
        ax2.grid(True, which="both", ls="-", alpha=0.5)
        
        # ??????? (????)
        threshold_val = 0.2 * M
        ax2.axhline(y=threshold_val, color='r', linestyle='--', label='Threshold (20%)')
        
        # ?????5%??????????????????? (????)
        ax2.axhline(y=0.05 * M, color='g', linestyle=':', label='Threshold (5%)')
        
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
        print("-> ?????????????")
        """
        
        # --- ??????????? ---
        fig, ax = plt.subplots(figsize=(8, 5))
        resp_array = np.array(resp).flatten() # ???1?????
        
        ax.plot(argm, resp_array, label='MUSIC Spectrum')
        ax.set_title(f'MUSIC Spectrum (N_ant={N_ant}, K={num_signals})')
        ax.set_xlabel('Angle [deg]')
        ax.set_ylabel('Spectrum Power')
        ax.set_yscale('log') # ????????????????????????
        ax.grid(True, which="both", ls="-", alpha=0.5)
        
        # ??????? (????)
        threshold_val = 0.2 * M
        ax.axhline(y=threshold_val, color='r', linestyle='--', label='Threshold (20%)')
        
        # ?????5%??????????????????? (????)
        ax.axhline(y=0.05 * M, color='g', linestyle=':', label='Threshold (5%)')
        
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        print("-> ?????????????")
        
    except Exception as e:
        print(f"-> ?????????????????: {e}")
        
    print("="*30 + "\n")

In [ ]:
def MUSIC_method_improved(Y, N_ant, num_signals):
    '''
    ???
    Y_music = np.mean(Y, axis=0)
    R_yy = Y_music@np.conjugate(np.transpose(Y_music))
    '''
    # ???: 
    # ?????(0)??????(2)??????????????????????
    # (N_trans, N_ant, N_sample) -> (N_ant, N_trans * N_sample) ???
    # ??????N_trans(?????)????????????????????
    # ????? R_yy ????????????
    
    N_trans, N_ant_data, N_sample = Y.shape
    # ???? (????? x ??????) ?2???????
    Y_music = Y.transpose(1, 0, 2).reshape(N_ant_data, -1)
    
    # ????????
    # (?????? ????? x ????? ???????????????????)
    R_yy = (Y_music @ np.conjugate(Y_music.T)) / Y_music.shape[1]
    
    # 1.?????
    eig_val, eig_vec = np.linalg.eig(R_yy)

    # 2.??????????????????
    idx = np.abs(eig_val).argsort()[::-1]
    # ??????????????????????????????????
    eig_val = eig_val[idx]
    eig_vec = eig_vec[:, idx]
    # ??????U???

    # num_signals?????????
    
    # ?????
    num_signals = 5
    
    U = eig_vec[:, num_signals:]

    # 3.???????val???????????????????
    resp = []
    argm = (np.arange(1800)-900)/100
    for val in argm:
        # ?????????????
        stv = stevec(N_ant, val*np.pi/180)
        # ??????????
        p = stv.T@U
        # L2?????????(pp^H)
        pp = p*np.conjugate(np.transpose(p))
        # A1????1?????????????
        # resp??????
        # ??????????????????????
        resp.append(1/(np.abs(pp)**2).A1)

    # 4.?????
    M = np.max(resp)
    
    # ?????
    #MUSIC_method_Debug(eig_val, N_ant, num_signals, resp, argm, M)
    
    est_ang = []
    # ????????????
    a, b = resp[0][0], resp[1][0]
    for i in range(1,len(argm)-1):
        c = resp[i+1][0]
        if a < b and b > c and b > 0.2 * M:
        #if a < b and b > c and b > 0.05 * M:
            est_ang.append(argm[i])
        a, b = b, c
    return est_ang

In [ ]:
def idx_to_xy(range_idx, angle_deg):
    """
    range_idx: cy_idxs_pred ?? (?: 496)
    angle_deg: MUSIC???????? [?] (?: est_ang[0])
    """
    p_bs = np.array([250, -18, 50])

    offset = 3411
    # (1) ????? [m]
    # ?????? = ?????? + ?????
    sample_idx = range_idx + offset
    # ???? = ?????? * Tc
    # ?? = ???? * c / 2
    R = (sample_idx * Tc * env.c) / 2

    # ??????????????
    height_diff = p_bs[2] - 1
    if R > height_diff:
        R_horizontal = np.sqrt(R**2 - height_diff**2)
    else:
        R_horizontal = 0

    # (2) ????? [rad]
    # ??????????????????(refangle)???
    # anglecal????: cy_ang = arctan(...) - refangle ???
    # arctan(...) = cy_ang + refangle
    theta_world = np.deg2rad(angle_deg) + refangle

    # (3) XY????? [m]
    # ???(p_bs)???????
    # cy_denum (dx) = R * cos(theta)
    # cy_nu (dy) = R * sin(theta)
    dx = R_horizontal * np.cos(theta_world)
    dy = R_horizontal * np.sin(theta_world)

    # ????(????)???
    # cy_denum = bs[0] - cy[0]  =>  cy[0] = bs[0] - dx
    # cy_nu    = cy[1] - bs[1]  =>  cy[1] = bs[1] + dy
    x = p_bs[0] - dx
    y = p_bs[1] + dy

    return x, y, R

In [ ]:
def DFT (N_trans, N_ant, rx_sample, Y, tx, N_sample, est_ang):
    # DFT???
    Dopp_dft = dft(N_trans)/np.sqrt(N_trans)
    Y_dft = np.zeros((N_trans, N_ant, rx_sample), dtype=np.complex128)
    for i in range(N_ant):
        Y_dft[:,i,:] = Dopp_dft@Y[:,i,:]
    val, res, rdresp = 0, [], np.zeros((N_trans,590), dtype=np.complex128)
    Q = 1

    # ??????????
    detected_objects = []
    
    # ????????????????
    rd_maps = []

    P_range = np.arange(590)+3411

    for i in range(len(est_ang)):
        current_angle = est_ang[i]
        #  ???????????????? rdresp_single ???
        rdresp_single = np.zeros((N_trans, 590), dtype=np.complex128)
        ang_rad = current_angle * np.pi / 180
        dup = np.ones((N_trans, 1, 1), dtype=np.complex128)
        
        # ???????????
        for p in P_range: # ?????????
            # p??????
            # p????SN?????????????????
            X_cand = np.zeros((N_ant, rx_sample), dtype=np.complex128)
            last = min(p+N_sample, rx_sample)
            X_cand[:,p:last] += tx[:,:last-p]
            g = np.conjugate(stevec(N_ant, ang_rad))@np.conjugate(np.transpose(stevec(N_ant, ang_rad)))@X_cand
            g_mat = dup*np.array(g)

            metric = np.sum(np.multiply(np.conjugate(Y_dft), g_mat), axis=(1,2))
            
            # ??? rdresp ???? rdresp_single ???
            rdresp_single[:, p-3411] = metric
            
        # === ??????: 0??????????????????????? ===
        if i == 0:  # ?????????
            P_range = np.arange(590) + 3411  # ????x?
            profile_power = np.abs(rdresp_single[0, :])
            
            plt.figure(figsize=(10, 4))
            plt.plot(P_range, profile_power)
            plt.title(f"[DFT] Range Profile at Angle {current_angle} deg (Doppler bin 0)")
            plt.xlabel("Range Index")
            plt.ylabel("Amplitude (Linear)")
            plt.grid(True)
            
            # ??????????????
            peak_val = np.max(profile_power)
            peak_db = 20 * np.log10(peak_val + 1e-10)
            print(f"? DFT Peak Value: {peak_val:.5f} ({peak_db:.2f} dB)")
            
            plt.show()
        # === ?????? ===
            
        # ?????????????????? (???????)
        rd_maps.append(rdresp_single.copy())
        rd_maps.append(i)
            
        # CFAR??????
        radar_in = torch.tensor(abs(rdresp_single)).unsqueeze(0).unsqueeze(0)
        detections, _, _ = ca_cfar_2d(radar_in, convert_from_db=False)
        idx_est = torch.nonzero(detections[0,0,0], as_tuple=True)[0].tolist()
        current_range_idxs, _ = thresholder(radar_in[0,0], idx_est)
        
        print("??????:", current_range_idxs)
        print("??????")
        
        # ?????????
        if len(current_range_idxs) > 0:
            print(f"?? {current_angle:.2f} ? ?????????:")
            for r_idx in current_range_idxs:
                x_est, y_est, r_est = idx_to_xy(r_idx, current_angle)
                detected_objects.append([r_est, current_angle ,x_est, y_est])
                print(f"  -> Index: {r_idx}, Range: {r_est:.2f}m, X: {x_est:.2f}m, Y: {y_est:.2f}m")
        else:
            print(f"????????")
    return detected_objects, rd_maps

#Y, phys_quantities = get_snapshot_data(100,0)
#est_ang = MUSIC_method(Y)
#detected_objects = DFT(N_trans, N_ant, rx_sample, Y, tx, N_sample, est_ang)
#print("Detected objects (X, Y):", detected_objects)

In [ ]:
def FFT(N_trans, N_ant, rx_sample, Y, tx, N_sample, est_ang, sym_duration, lam):
    # ???????
    Dopp_dft = dft(N_trans)/np.sqrt(N_trans)
    Y_dft = np.zeros((N_trans, N_ant, rx_sample), dtype=np.complex128)
    for i in range(N_ant):
        Y_dft[:,i,:] = Dopp_dft@Y[:,i,:]

    detected_objects = []
    rd_maps = []

    P_range = np.arange(590)+3411

    # FFT?????????
    tx_arr = np.array(tx) 
    n_fft = 2**(int(rx_sample + N_sample).bit_length())
    
    # ????????FFT??
    TX_F_all = fft(tx_arr, n=n_fft, axis=1) 
    Y_F_all = fft(Y_dft, n=n_fft, axis=2)

    for i in range(len(est_ang)):
        current_angle = est_ang[i]
        ang_rad = current_angle * np.pi / 180
        
        # ?????????? (N_ant, 1)
        s_vec = np.array(stevec(N_ant, ang_rad))
        
        # --- ???????????????? ---
        
        # ???????: Y_combined = sum(Y * s)
        # s_vec ? (1, N_ant, 1) ???????
        # Y_F_all: (N_trans, N_ant, n_fft)
        # ?????????????????? -> (N_trans, n_fft)
        # ???
        #w_vec = np.conjugate(s_vec).reshape(1, N_ant, 1)
        #Y_F_combined = np.sum(Y_F_all * w_vec, axis=1)
        # ???
        w_vec = s_vec.reshape(1, N_ant, 1) 
        Y_F_combined = np.sum(Y_F_all * w_vec, axis=1)

        # ????(????)???: X_combined = sum(X * s*)
        # ???? g = s* s^H X ???X??????? s^H (??? s???)
        # TX_F_all: (N_ant, n_fft)
        s_vec_conj_reshape = np.conjugate(s_vec).reshape(N_ant, 1)
        TX_F_combined = np.sum(TX_F_all * s_vec_conj_reshape, axis=0) # -> (n_fft,)

        # --- ???? (Correlation) ---
        # Correlation = IFFT( FFT(Y) * conj(FFT(X)) )
        # Broadcasting: (N_trans, n_fft) * (n_fft,)
        CORR_f = Y_F_combined * np.conjugate(TX_F_combined)
        
        # ???????
        corr_time = ifft(CORR_f, axis=1)

        # --- ?????????? ---
        rdresp_single = corr_time[:, P_range]
        """
        if i == 0:
            print("\n--- [Phase 4] Time Domain / Range Profile (i=0) ---")
            print(f"rdresp_single shape (Trans, RangeBins): {rdresp_single.shape}")
            
            # ??????????
            # ??????(Trans)?0??(?????)?????????????
            profile_power = np.abs(rdresp_single[0, :]) 
            plt.figure(figsize=(10, 4))
            plt.plot(P_range, profile_power)
            plt.title(f"Range Profile at Angle {current_angle} deg (Doppler bin 0)")
            plt.xlabel("Range Index")
            plt.ylabel("Amplitude")
            plt.grid(True)
            plt.show()
            print("? ??????????????????????????????")
        """
        # --- ???? ---
        rd_maps.append(rdresp_single.copy())
        rd_maps.append(i)

        # --- ??? ---
        """
        # CFAR??????
        radar_in = torch.tensor(abs(rdresp_single)).unsqueeze(0).unsqueeze(0)
        detections, _, _ = ca_cfar_2d(radar_in, convert_from_db=False)
        idx_est = torch.nonzero(detections[0,0,0], as_tuple=True)[0].tolist()
        current_range_idxs, _ = thresholder(radar_in[0,0], idx_est)

        print("??????:", current_range_idxs)
        
        # ?????????
        if len(current_range_idxs) > 0:
            print(f"?? {current_angle:.2f} ? ?????????:")
            for r_idx in current_range_idxs:
                x_est, y_est, r_est = idx_to_xy(r_idx, current_angle)
                detected_objects.append([r_est, current_angle ,x_est, y_est])
                print(f"  -> Index: {r_idx}, Range: {r_est:.2f}m, X: {x_est:.2f}m, Y: {y_est:.2f}m")
        else:
            print(f"????????")
        """
        # --- ???? ---

        # --- ??? ---
        # --- ???????????? ---

        # 1. CFAR??
        # radar_in??? (Batch, Channel, Height, Width)
        radar_in = torch.tensor(np.abs(rdresp_single)).unsqueeze(0).unsqueeze(0)
        detections_bool = ca_cfar_improved(radar_in, convert_from_db=False)

        # 2. dB?????? (Thresholder?)
        radar_db = 20 * torch.log10(radar_in[0,0])

        # 3. ???? (??????????????)
        # detections_bool[0,0] ??????2??????????
        # tar_thres ????????? (-30 ? -25 ??)
        raw_targets = thresholder_improved(radar_db, detections_bool[0,0], tar_thres=-25)

        # 4. ???? (Cleansing)
        clean_targets = cleansing_improved(raw_targets, dist_tol=5, dop_tol=2)

        # 5. ?????
        if len(clean_targets) > 0:
            print(f"?? {current_angle:.2f} ? ?????????:")
            for tgt in clean_targets:
                d_idx, r_idx, power = tgt
                
                # ??????? (?? sym_duration, lam ??????)
                r_est, v_est, x_est, y_est = convert_to_physical(
                    d_idx, r_idx, N_trans, sym_duration, lam, current_angle, idx_to_xy
                )
                
                #detected_objects.append([r_est, v_est, current_angle, x_est, y_est]) # v_est???
                # ????? power ?????????? (?????)
                detected_objects.append([r_est, current_angle, x_est, y_est, v_est, power])
                print(f"  -> Index: ({d_idx}, {r_idx}), Power: {power:.1f}dB, Range: {r_est:.2f}m, Vel: {v_est:.2f}m/s")
        else:
            print("????????")
        # --- ???? ---
    # ???? ?????????????????????????
    final_objects = remove_ghosts_across_angles(detected_objects)    
        
    return final_objects, rd_maps

In [ ]:
def RD_MAP(rd_maps, frame_id, Sym_duration, save_dir="./rd_maps/"):
    l_speed = 299792458
    Tc = 0.509*1e-9 
    f_carrier = 28 * 1e+9

    lam = l_speed / f_carrier
    
    
    # ?????????????
    os.makedirs(save_dir, exist_ok=True)
    
    rdresp_list = np.array(rd_maps[::2])
    #idx = np.array(rd_maps[1::2])
    
    # ????? (m)
    range_res = l_speed * Tc / 2.0
    print(range_res)
    # ?? (??):
    # rdresp?0??? p-3411 = 0 => p = 3411 (??????????)
    start_sample_idx = 3411
    num_range_bins = 590

    r_min = start_sample_idx * range_res
    r_max = (start_sample_idx + num_range_bins) * range_res
    
    v_range = lam / (2 * Sym_duration)  # ??????

    v_min_val = -v_range / 2
    v_max_val = v_range / 2
    
    i = 0

    for rd_resp in rdresp_list:
        #output = rd_resp[:100,:] + 1e-10
        #output = rd_resp + 1e-10
        output = np.fft.fftshift(rd_resp, axes=0) + 1e-10
        plt.imshow(20*np.log10(np.abs(output)), aspect='auto', interpolation='nearest',
             extent = [r_min, r_max, v_min_val, v_max_val], origin='lower', vmin=-30, vmax = -10)
        plt.xlabel('Range [m]')
        plt.ylabel('Velocity [m/s]') # ?: DFT???????????????????????
        plt.colorbar(label='Power [dB]')
        plt.title('Range-Doppler Map')
        plt.grid(False) # ??
        plt.xlim(280,300)
        plt.ylim(-30,30)
        # --- ???? ---
        # ???????? (?: ./rd_maps/rd_map_12345.png)
        save_path = os.path.join(save_dir, f"rd_map_{frame_id}_for_{i}.png")
        plt.savefig(save_path)
        print(f"Saved: {save_path}")
        
        # --- ?????? ---
        # show() ???? close() ???????????????????????????
        # ????????? plt.show() ??? plt.savefig() ????????
        plt.close()
        i += 1
    
    # ????
    

In [ ]:
def calc_velocity(doppler_idx, N_trans, sym_duration, lam):
    """
    doppler_idx: CFAR???????????????? (0 ~ N_trans-1)
    N_trans:     ????? (FFT??)
    sym_duration: 1???????????? (PRI) = rx_sample * Tc
    lam:         ?? (c / f_carrier)
    """
    # 1???????????
    # ???2???(2R)?????????????
    velocity_res = lam / (2.0 * sym_duration * N_trans)
    
    # ?????? (FFT????????)
    if doppler_idx < N_trans / 2:
        # ???? (????) ???????????
        velocity = doppler_idx * velocity_res
    else:
        # ???? (???) 
        # ?: N=500? idx=499 ?? -1 * res
        velocity = (doppler_idx - N_trans) * velocity_res
        
    return velocity

In [ ]:
# csv???????????????
def syntax_check(num, idx, y_value, velocity_value, x_pos):
    idx_list = np.zeros(num, dtype=np.int32)
    position, velocity = np.zeros((num, 3)), np.zeros((num, 3))
    i=0
    s = set()
    while i < num:
        # ?????????????????????
        if idx+10 in s or idx-10 in s or idx in s: continue
        for j in range(-20,20): s.add(idx+j)
        
        idx_list[i] = idx
        position[i] = np.array([x_pos[idx], y_value, 1])
        # v_cy[i] = np.array([np.random.rand(1)[0]*5.56, 0, 0])
        velocity[i] = np.array([velocity_value, 0, 0])
        
        i+=1
    return idx_list, position, velocity

def get_snapshot_data(cy_idx_target, ve_idx_target,x_pos, p_bs, tx, cy_v_value, ve_v_value, num_cy, num_ve, num_rp, sym_duration):
    # Cyclist????
    cy_idx = np.zeros(num_cy, dtype=np.int32)
    p_cy, v_cy = np.zeros((num_cy, 3)), np.zeros((num_cy, 3))
    # ???????????????
    cy_idx, p_cy, v_cy = syntax_check(num_cy, cy_idx_target, 15, cy_v_value, x_pos)
    n_cy = len(p_cy)
    
    # Vehicle????
    ve_idx = np.zeros(num_ve, dtype=np.int32)
    p_ve, v_ve = np.zeros((num_ve, 3)), np.zeros((num_ve, 3))
    # ???????????????
    ve_idx, p_ve, v_ve = syntax_check(num_ve, ve_idx_target, 7.5, ve_v_value, x_pos)
    n_ve = len(p_ve)

    # ramppost????
    p_rp, v_rp = np.zeros((num_rp, 3)), np.zeros((num_rp, 3))
    n_rp = len(p_rp)
    
    # csv?????????
    cy_idx += 1
    ve_idx += 1

    P_rx_cy, tstemp_rx_cy, phase_cy, P_rx_ve, tstemp_rx_ve, phase_ve, P_rx_rp, tstemp_rx_rp, phase_rp = [], [], [], [], [], [], [], [], []
    # 3??????????????(??????????????)
    for i in range(len(cy_idx)):
        df = pd.read_csv("./ped/0.5nano/complex-impulse-response-Run{:04d}-Sensor_0_Tx_0_to_Rx_0.csv".format(cy_idx[i]))
        tstemp_rx_cy.append(np.ceil((df["Time (s)"]/Tc).values[3:]))                            #?????????????????????????
        P_rx_cy.append(df["| Total Complex Impulse Response total | (W)"].values[3:])
        phase_cy.append(df['Phase( Total Complex Impulse Response total ) (rad)'].values[3:])
    for i in range(len(ve_idx)):
        df = pd.read_csv("./vehicle/0.5nano/complex-impulse-response-Run{:04d}-Sensor_0_Tx_0_to_Rx_0.csv".format(ve_idx[i]))
        tstemp_rx_ve.append(np.ceil((df["Time (s)"]/Tc).values[3:]))
        P_rx_ve.append(df["| Total Complex Impulse Response total | (W)"].values[3:])
        phase_ve.append(df['Phase( Total Complex Impulse Response total ) (rad)'].values[3:])
    for i in range(1,4):
        df = pd.read_csv("./Ramposts/"+str(i)+"/complex-impulse-response-Run0001-Sensor_0_Tx_0_to_Rx_0.csv")
        tstemp_rx_rp.append(np.ceil((df["Time (s)"]/Tc).values[5:]))
        P_rx_rp.append(df["| Total Complex Impulse Response total | (W)"].values[5:])
        phase_rp.append(df['Phase( Total Complex Impulse Response total ) (rad)'].values[5:])
        
    phys_quantities = env.phys_quantities(p_bs, p_cy, v_cy, n_cy, p_ve, v_ve, n_ve, p_rp, v_rp, n_rp)
    real_cy_coordinates = []
    real_vel_coordinates = []
    for i in range(len(p_cy)):
        real_cy_coordinates.append(np.array([p_cy[i][0], p_cy[i][1]]))
    for i in range(len(p_ve)):
        real_vel_coordinates.append(np.array([p_ve[i][0], p_ve[i][1]]))
        
    ######### We need some functions here. it is like this. read the csv. this outputs the time stemp and corresponding rx value.
    # P_rx_cy_dB = -114
    #P_rx_cy_dB = -111
    #P_rx_ve_dB = -114
    #P_rx_cy_dB = 0
    #P_rx_ve_dB = 0
    
    P_N_dB = -81 # target noise power when using 2GHz BW
    # === ????SNR?????????? ===
    print(f"\n--- SNR Debug Info (Noise Floor: {P_N_dB} dB) ---")
    # Cyclist?SNR??
    if len(P_rx_cy) > 0:
        for i, power_array in enumerate(P_rx_cy):
            # ?????????????(????)??????????
            sig_watt = np.max(power_array)
            if sig_watt > 0:
                sig_db = 10 * np.log10(sig_watt)
                snr = sig_db - P_N_dB
                print(f"[Cyclist {i}] Signal: {sig_db:.2f} dB | Input SNR: {snr:.2f} dB")
            else:
                print(f"[Cyclist {i}] Signal Power is 0 or invalid.")

    Y = env.rx_multiple(tx, P_rx_cy, tstemp_rx_cy, phase_cy, P_rx_ve, tstemp_rx_ve, phase_ve, P_rx_rp, tstemp_rx_rp, phase_rp, P_N_dB, sym_duration, N_trans)
    
    return Y, phys_quantities, real_cy_coordinates, real_vel_coordinates

#x_pos = list(np.arange(-450, -149)/10)
#p_bs = np.array([250, -18, 50])
#Y, phys_quantities = get_snapshot_data(100,0,x_pos,p_bs)

In [ ]:
###### symbol time & carrier frequency ######
T_symbol = 1.115 * 1e-6              # ??????????????

T_OFDM = 1.0425 * 1e-6
f_carrier = 28 * 1e+9               # ??????(?????????????????)
Tc = 0.509*1e-9                      # ????????(??????????????????????????????)

###### tx/rx ######
N_ant = 16                                      # ??????
BW = 1.966080e+9  
BW_sub = BW/N_ant                             # ??????????(???????????????)  
N_sample = int(np.floor(T_symbol/Tc))           # ?????????????????(?????????????)
offset = 4000
rx_sample = offset + N_sample                   # ?????????????????(?????????)

###### Radar setting ######
refangle = np.arctan(33/265)/2
N_chirp = 446                    # ??????
mu = BW_sub/T_symbol * 0.98      # ???????

Q = 30                           # ??????
Phi = 0.3*np.pi                  # ??????

l_speed = 299792458              # ??

#N_trans = 100                    # ?????DFT???(?????????????????????????)
N_trans = N_chirp


lam = l_speed / f_carrier        # ??
T_sensing = N_chirp * T_symbol + offset * Tc
print("?????????????:", T_sensing)
#sym_duration = (4000 + int(np.floor(1.115e-6 / 0.509e-9))) * 0.509e-9
sym_duration = N_chirp * N_sample * Tc + offset * Tc

env = cyclist_env(f_carrier, N_ant, BW, BW_sub, N_sample, rx_sample, Tc, mu, l_speed)
# ???????(??:???????:?????)
tx = env.tx()

# ??????????
p_bs = np.array([250, -18, 50])                         #??????

# ?????
num_cy = 1
num_ve = 0
num_rp = 0
# ?????????????
#start_cy_idx = 135
#start_ve_idx = 150
start_cy_idx = 75
start_ve_idx = 50

#end_cy_idx = 300
#end_ve_idx = 3
end_cy_idx = 135
end_ve_idx = 150

cy_interval = 3
ve_interval = 5

cy_v_value = 6
ve_v_value = 10

cy_idx = start_cy_idx
ve_idx = start_ve_idx

measurement_history = []
frame_count = 0

directory_path = "./rd_maps"
try:
    # ????????????????(?????????)???
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)

        # ?????????????
        if os.path.isfile(full_path):
            # ???????
            os.remove(full_path)
        
        # ????????????????
        elif os.path.isdir(full_path):
            print(f"?? ?????????????????: {full_path}")
            # ????????????????????????
            # ??????????????????????????
            # (?: delete_all_files_in_directory(full_path))

except FileNotFoundError:
    print(f"???: ?????????????? - {directory_path}")
except Exception as e:
    print(f"??????????????????: {e}")

real_cy_coordinates_all = []
real_cy_ranges_all = []
real_cy_angles_all = []
real_cy_velocities_all = []
real_ve_coordinates_all = []
real_ve_ranges_all = []
real_ve_angles_all = []
real_ve_velocities_all = []

# ??????
while cy_idx <= end_cy_idx and ve_idx <= end_ve_idx:
    x_pos = list(np.arange(-450, -149)/10)                  #x??????
    
    Y, phys_quantities, real_cy_coordinates, real_vel_coordinates = get_snapshot_data(cy_idx,ve_idx,x_pos,p_bs, tx, cy_v_value, ve_v_value, num_cy, num_ve, num_rp, sym_duration)

    if num_cy > 0:
        real_cy_coordinates_all.append(real_cy_coordinates)
        real_cy_ranges_all.append(phys_quantities["range"]["cyclists"][0])
        real_cy_angles_all.append(phys_quantities["angle"]["cyclists"][0])
        real_cy_velocities_all.append(phys_quantities["relative_velocity"]["cyclists"][0])
    if num_ve > 0:
        real_ve_coordinates_all.append(real_vel_coordinates)
        real_ve_ranges_all.append(phys_quantities["range"]["vehicles"][0])
        real_ve_angles_all.append(phys_quantities["angle"]["vehicles"][0])
        real_ve_velocities_all.append(phys_quantities["relative_velocity"]["vehicles"][0])
    #est_ang = MUSIC_method(Y, N_ant)
    est_ang = MUSIC_method_improved(Y, N_ant, num_cy + num_ve + num_rp)
    #est_ang = Root_MUSIC_method(Y, N_ant, num_cy + num_ve + num_rp)
    
    #detected_objects, rd_maps = DFT(N_trans, N_ant, rx_sample, Y, tx, N_sample, est_ang)
    detected_objects, rd_maps = FFT(N_trans, N_ant, rx_sample, Y, tx, N_sample, est_ang, sym_duration, lam)
    #plot_rd_maps(rd_maps, est_ang)
    RD_MAP(rd_maps, frame_count, sym_duration, save_dir="./rd_maps")
    
    if len(detected_objects) > 0:
        for obj in detected_objects:
            measurement_history.append({
                "Time": frame_count,  # ??(??????)
                "Range": obj[0],      # ?? [m]
                "Angle": obj[1],      # ?? [?]
                "X": obj[2],          # X?? [m]
                "Y": obj[3],          # Y?? [m]
                "Velocity": obj[4]    # ?? [m/s]
            })
    else:
        # ????????????????????????????????NaN????
        # (????????else?????????????)
        measurement_history.append({
            "Time": frame_count,
            "Range": np.nan,
            "Angle": np.nan,
            "X": np.nan,
            "Y": np.nan,
            "Velocity": np.nan
        })
    
    cy_idx += cy_interval
    ve_idx += ve_interval
    frame_count += 1
    
# csv???????
# --- 6. CSV???? ---
df_measurements = pd.DataFrame(measurement_history)

# ??
csv_filename = "measurements.csv"
df_measurements.to_csv(csv_filename, index=False)

print("\nData generation complete.")
print(f"Saved '{csv_filename}'.")

real_positions = []
csv_filename = "true_information.csv"
for i in range(max(len(real_cy_coordinates_all), len(real_ve_coordinates_all))):
    if num_cy > 0:
        real_positions.append({
            "Time": i,
            "target_id":0,
            "X": real_cy_coordinates_all[i][0][0],
            "Y": real_cy_coordinates_all[i][0][1],
            "Range": real_cy_ranges_all[i],
            "Angle": real_cy_angles_all[i],
            "Velocity": real_cy_velocities_all[i],
        })

    if num_ve > 0:
        real_positions.append({
            "Time": i,
            "target_id":1,
            "X": real_ve_coordinates_all[i][0][0],
            "Y": real_ve_coordinates_all[i][0][1],
            "Range": real_ve_ranges_all[i],
            "Angle": real_ve_angles_all[i],
            "Velocity": real_ve_velocities_all[i]
        })
df_real_positions = pd.DataFrame(real_positions)
df_real_positions.to_csv(csv_filename, index=False)
print(f"Saved true positions to '{csv_filename}'.")

In [ ]:
import imageio
import glob
import os
import math

def create_split_gifs(image_folder, output_prefix, frames_per_gif=100, duration=0.1):
    """
    ?????????GIF???????????
    
    image_folder: ??????
    output_prefix: ??????????? (?: "radar_movie") -> "radar_movie_part1.gif" ?????
    frames_per_gif: 1??GIF????????? (i?????????????GIF?????)
    duration: ??????
    """
    
    # 1. ?????????
    search_path = os.path.join(image_folder, "*.png")
    files = glob.glob(search_path)
    
    if not files:
        print("??????????????")
        return

    # 2. ????? "rd_map_{id}_for_{i}.png" ? {i} ???????
    def get_sort_key(filepath):
        try:
            filename = os.path.basename(filepath)
            # "_for_" ?????????
            return int(filename.split('_for_')[-1].replace('.png', ''))
        except:
            return 0

    files.sort(key=get_sort_key)
    
    total_files = len(files)
    print(f"?? {total_files} ?????????????")
    print(f"{frames_per_gif} ????????GIF???????")

    # 3. ????GIF??
    # 0??total_files???frames_per_gif ???????????
    num_parts = math.ceil(total_files / frames_per_gif)
    
    for part_idx, start_idx in enumerate(range(0, total_files, frames_per_gif)):
        # ??????????????????????
        end_idx = start_idx + frames_per_gif
        batch_files = files[start_idx : end_idx]
        
        images = []
        for filename in batch_files:
            images.append(imageio.imread(filename))
            
        # ???????? (?: radar_movie_part01.gif)
        # part_idx + 1 ??? 1???????????
        output_name = f"{output_prefix}_part{part_idx + 1:02d}.gif"
        
        imageio.mimsave(output_name, images, duration=duration, loop=0)
        
        # ????
        first_i = get_sort_key(batch_files[0])
        last_i = get_sort_key(batch_files[-1])
        print(f"[{part_idx + 1}/{num_parts}] Saved: {output_name} (i={first_i} to {last_i})")

# --- ???? ---
target_folder = "./rd_maps/" 

# ?????????1??GIF???????????
# ?: 50????????? 50 ???
create_split_gifs(
    image_folder=target_folder, 
    output_prefix="rd_map_animation", 
    frames_per_gif=50,  # <--- ????????
    duration=0.1
)